In [28]:
import gymnasium as gym
import numpy as np

In [29]:
# -------------------------------------------------
# Create FrozenLake Environment
# -------------------------------------------------

env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
env = env.unwrapped

n_states = env.observation_space.n
n_actions = env.action_space.n

gamma = 0.99
theta = 1e-8


In [30]:
# -------------------------------------------------
# Helper Function: One-Step Lookahead
# -------------------------------------------------

def one_step_lookahead(env, state, V, gamma):
    """
    Calculates the expected value of each action
    from a given state.
    """

    action_values = np.zeros(n_actions)

    for action in range(n_actions):
        for probability, next_state, reward, terminated in env.P[state][action]:
            action_values[action] += probability * (
                reward + gamma * V[next_state] * (not terminated)
            )

    return action_values

In [31]:
# -------------------------------------------------
# Policy Evaluation
# -------------------------------------------------

def policy_evaluation(policy, env, gamma, theta, max_iter=1000):
    """
    Evaluate a given policy by iteratively updating the state-value function.
    """
    V = np.zeros(n_states)

    for _ in range(max_iter):
        delta = 0.0

        for state in range(n_states):
            v_state = 0.0

            for action, action_prob in enumerate(policy[state]):
                for probability, next_state, reward, terminated in env.P[state][action]:
                    v_state += action_prob * probability * (
                        reward + gamma * V[next_state] * (not terminated)
                    )

            delta = max(delta, abs(v_state - V[state]))
            V[state] = v_state

        if delta < theta:
            break
    return V

In [32]:
# -------------------------------------------------
# Policy Improvement
# -------------------------------------------------

def policy_improvement(env, V, gamma):
    """
    Improve the policy greedily with respect to the current value function.
    """
    policy = np.zeros((n_states, n_actions))

    for state in range(n_states):
        action_values = one_step_lookahead(env, state, V, gamma)
        best_action = np.argmax(action_values)
        policy[state, best_action] = 1.0

    return policy

In [36]:
# -------------------------------------------------
# Policy Iteration
# -------------------------------------------------

def policy_iteration(env, gamma, theta, max_iterations=1000):
    """
    Run policy iteration until the policy converges.
    """
    policy = np.ones((n_states, n_actions)) / n_actions
    V = np.zeros(n_states)
    print("-------------------------------------------------")
    print("Before Policy Iteration:")
    print("-------------------------------------------------")
    print_value_function(V)

    for iteration in range(max_iterations):
        V = policy_evaluation(policy, env, gamma, theta)
        new_policy = policy_improvement(env, V, gamma)

        if np.allclose(policy, new_policy):
            print(f"Policy iterations: {iteration + 1}")
            print("-------------------------------------------------")
            print("After Policy Iteration :")
            print("-------------------------------------------------")
            print_value_function(V)
            print_policy(policy)
            return new_policy, V

        policy = new_policy
        print(f"Policy iterations: {iteration +1}")
        print_value_function(V)
        print_policy(policy)
        print()

    print_value_function(V)
    return policy, V

In [37]:

# -------------------------------------------------
# Display Functions
# -------------------------------------------------

def print_value_function(V):
    print("\nOptimal State-Value Function:")
    print(np.round(V.reshape(4, 4), 4))


def print_policy(policy):
    action_symbols = {
        0: "←",
        1: "↓",
        2: "→",
        3: "↑"
    }

    best_actions = np.argmax(policy, axis=1)
    policy_grid = np.array(
        [action_symbols[action] for action in best_actions]
    ).reshape(4, 4)


    print("\nOptimal Policy:")
    print(policy_grid)



In [38]:
# -------------------------------------------------
# Run Policy Iteration
# -------------------------------------------------

optimal_policy, optimal_value_function = policy_iteration(
    env,
    gamma=gamma,
    theta=theta
)

print("Name : SANJAY C")
print("Register Number : 212223240150")
print_value_function(optimal_value_function)
print_policy(optimal_policy)

env.close()

-------------------------------------------------
Before Policy Iteration:
-------------------------------------------------

Optimal State-Value Function:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Policy iterations: 1

Optimal State-Value Function:
[[0.0124 0.0104 0.0193 0.0095]
 [0.0148 0.     0.0389 0.    ]
 [0.0326 0.0843 0.1378 0.    ]
 [0.     0.1703 0.4336 0.    ]]

Optimal Policy:
[['←' '↑' '←' '↑']
 ['←' '←' '←' '←']
 ['↑' '↓' '←' '←']
 ['←' '→' '↓' '←']]

Policy iterations: 2

Optimal State-Value Function:
[[0.5325 0.4498 0.3807 0.3695]
 [0.5486 0.     0.3232 0.    ]
 [0.5814 0.6318 0.5987 0.    ]
 [0.     0.7344 0.8592 0.    ]]

Optimal Policy:
[['←' '↑' '↑' '↑']
 ['←' '←' '←' '←']
 ['↑' '↓' '←' '←']
 ['←' '→' '↓' '←']]

Policy iterations: 3
-------------------------------------------------
After Policy Iteration :
-------------------------------------------------

Optimal State-Value Function:
[[0.542  0.4988 0.4707 0.4569]
 [0.5585 0.     0.3583 0.    ]
